Assignment:
1. Generate Alpha Strategy returns (random)
2. Calculate the Covariance matrix
3. Run Mean-Variance Optimization (In-Sample) and compare with Equal-Weight and Risk Parity

In [18]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.optimize import minimize

ANN = 252  # annualization factor (business days / year)
# random PnL shocks (5 strategies, distinct vols)
N_STRATEGIES = 5
# generate vols for each strategy
vols = np.random.uniform(0.012, 0.025, N_STRATEGIES)

### Generate 5 Alpha strategies

In [2]:
# UTC-aware datetime index (daily business days)
start_date = pd.Timestamp("2024-01-01", tz="UTC")
todays_date = pd.Timestamp.now(tz="UTC")
idx = pd.date_range(start=start_date, end=todays_date, freq="B", tz="UTC")

mean_perf = -np.inf
while mean_perf < 0.25:    # ensure mean performance > threshold
    random_gen = np.random.default_rng()
    period_pnl = random_gen.standard_normal((len(idx), N_STRATEGIES)) * vols

    # Cumulative PnL curves (equity from trading each strategy)
    pnl_strategies = pd.DataFrame(
        np.cumsum(period_pnl, axis=0),
        index=idx,
        columns=[f"Alpha_{i + 1}" for i in range(N_STRATEGIES)],
    )

    mean_perf = pnl_strategies.iloc[-1].mean()

pnl_strategies.index.name = "Date"

fig = px.line(pnl_strategies, y=pnl_strategies.columns, x=pnl_strategies.index, title="PnL Curves of Alpha Strategies")
fig.update_layout(
    xaxis_title=None,
    yaxis_title="PnL",
    legend_title="Alpha Strategy",
    title_font_size=24,
    title_font_color="black"
)
fig.show()

### Alpha covariances

In [3]:
# alpha covariance matrix
R = pnl_strategies.diff().dropna()

cov = R.cov()
corr = R.corr()
print("Covariance matrix of period PnL (alphas):")
display(cov.round(6))

print("Correlation matrix of period PnL (alphas):")
display(corr.round(3))


Covariance matrix of period PnL (alphas):


,Alpha_1,Alpha_2,Alpha_3,Alpha_4,Alpha_5
Alpha_1,0.000503,-0.000029,0.000001,-0.000018,0.000030
Alpha_2,-0.000029,0.000591,-0.000038,-0.000010,0.000006
Alpha_3,0.000001,-0.000038,0.000531,0.000003,-0.000011
Alpha_4,-0.000018,-0.000010,0.000003,0.000156,-0.000006
Alpha_5,0.000030,0.000006,-0.000011,-0.000006,0.000526


Correlation matrix of period PnL (alphas):


,Alpha_1,Alpha_2,Alpha_3,Alpha_4,Alpha_5
Alpha_1,1.000,-0.052,0.002,-0.064,0.059
Alpha_2,-0.052,1.000,-0.068,-0.033,0.011
Alpha_3,0.002,-0.068,1.000,0.011,-0.021
Alpha_4,-0.064,-0.033,0.011,1.000,-0.019
Alpha_5,0.059,0.011,-0.021,-0.019,1.000


### Optimization: Mean-Variance vs Equal-Weight vs Risk Parity

In [4]:
# equal weights
w_eq = np.ones(N_STRATEGIES) / N_STRATEGIES
w_eq

array([0.2, 0.2, 0.2, 0.2, 0.2])

In [5]:
# risk parity weights
inv_vol = 1.0 / (R.std(ddof=1))
w_rp = inv_vol.values / inv_vol.values.sum()
w_rp

array([0.17634618, 0.16269193, 0.17162926, 0.31690296, 0.17242967])

In [6]:
def sharpe(period_pnl: np.ndarray) -> float:
    p = np.asarray(period_pnl, dtype=float)
    return float(np.sqrt(ANN) * p.mean() / (p.std(ddof=1)))

# negate Sharpe ratio to minimize
def neg_portfolio_sharpe(w: np.ndarray) -> float:
    w = np.asarray(w, dtype=float)
    port = R.values @ w
    return -sharpe(port)

# Mean-Variance Optimization: maximize Sharpe ratio with constraints 0<weights<1 and sum(weights) = 1
res = minimize(
    neg_portfolio_sharpe,
    w_eq,
    method="SLSQP",
    bounds=[(0.0, 1.0)] * N_STRATEGIES,
    constraints=({"type": "eq", "fun": lambda w: float(np.sum(w) - 1.0)},),
    options={"ftol": 1e-12},
)
w_max_sharpe = res.x / res.x.sum()
w_max_sharpe

array([0.07386434, 0.58250648, 0.17164077, 0.        , 0.17198841])

### Portfolio Performances

In [12]:
idx_names = list(pnl_strategies.columns)
weight_df = pd.DataFrame(
    {"equal_weight": w_eq, "max_sharpe_mvo": w_max_sharpe, "risk_parity_inv_vol": w_rp},
    index=idx_names,
)

rows = []
for c in idx_names:
    rows.append(
        {
            "Alpha/Ptf": c,
            "Total PnL": float(pnl_strategies[c].iloc[-1]),
            "Sharpe": sharpe(R[c].values),
        }
    )

perfs = []
for label, w in (
    ("Equal Weight", w_eq),
    ("Max Sharpe MVO", w_max_sharpe),
    ("Risk Parity", w_rp),
):
    port_pnl = R.values @ w
    perfs.append(pd.DataFrame(port_pnl.cumsum(), index=idx[1:], columns=[label]))
    rows.append(
        {
            "Alpha/Ptf": label,
            "Total PnL": float(port_pnl.sum()),
            "Sharpe": sharpe(port_pnl),
        }
    )

summary = pd.DataFrame(rows).set_index("Alpha/Ptf")
print("Final PnL and Sharpe (annualized, daily period PnL):")
display(summary.round(6))


Final PnL and Sharpe (annualized, daily period PnL):


,Total PnL,Sharpe
Alpha/Ptf,,
Alpha_1,0.098253,0.098269
Alpha_2,1.038535,1.179156
Alpha_3,0.195813,0.247895
Alpha_4,-0.124232,-0.259957
Alpha_5,0.305733,0.350680
Equal Weight,0.306051,0.889242
Max Sharpe MVO,0.710912,1.279388
Risk Parity,0.236356,0.749434


In [16]:
# plot portfolio performances
portfolio_performances = pd.concat(perfs, axis=1)
fig = px.line(
    portfolio_performances,
    y=portfolio_performances.columns,
    x=portfolio_performances.index,
    title="Portfolio Performances",
)
fig.update_layout(
    xaxis_title=None,
    yaxis_title="PnL",
    legend_title="Portfolio Weights",
    title_font_size=24,
    title_font_color="black"
)
fig.update_yaxes(tickformat=".0%")
fig.show()


In [32]:
# scatterplot of Alpha strategies and portfolio performances on x = Vol, y = Total PnL

# Alpha Vols and PnLs
vols = pnl_strategies.std(axis=0)
pnls = pnl_strategies.iloc[-1]

# Portfolio Vols and PnLs
ptf_vols = portfolio_performances.std(axis=0)
ptf_pnls = portfolio_performances.iloc[-1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=vols, y=pnls, mode='markers', name='Alpha Strategies'))
fig.add_trace(go.Scatter(x=ptf_vols, y=ptf_pnls, mode='markers', name='Portfolio'))
fig.update_layout(
    xaxis_title="Volatility",
    yaxis_title="Total PnL",
    title="Alpha Strategies and Portfolio Performances"
)
fig.show()